In [1]:
# ==============================================================================
# Package Installation
# ==============================================================================
!pip install -q --upgrade \
    "transformers>=4.44.0" \
    "peft>=0.12.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.33.0" \
    "pyarrow>=14.0.0" \
    "datasets>=2.20.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
# ==============================================================================
# STELLAR-ARM 4-WAY FAILURE-MODE AUDIT (FAST LENGTH-SORTED + LIVE PROGRESS)
# Evaluates ONLY scalar_logit (actual RM reward) for apples-to-apples comparison
# ==============================================================================
import os
import gc
import time
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Any
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, set_peft_model_state_dict

In [3]:
# ------------------------------------------------------------------------------
# 1. AUTO-DISCOVER BASE MODEL, TEST PARQUET, AND ALL 4 CHECKPOINTS
# ------------------------------------------------------------------------------
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
if not os.path.exists(MODEL_PATH):
    for root, _, files in os.walk("/kaggle/input"):
        if "config.json" in files and "qwen" in root.lower() and "stellar" not in root.lower():
            MODEL_PATH = root
            break
print(f"[BASE MODEL] {MODEL_PATH}")

def find_test_benchmark_parquet() -> Optional[str]:
    preferred = [
        "/kaggle/input/notebooks/pradeep01223/stellar-arm-shard-data-pipeline-s1/arm_test_benchmark.parquet",
        "/kaggle/input/stellar-arm-shard-data-pipeline-s1/arm_test_benchmark.parquet",
        "/kaggle/working/arm_test_benchmark.parquet",
    ]
    for p in preferred:
        if os.path.exists(p):
            return p
    for search_root in ["/kaggle/input", "/kaggle/working", "."]:
        if not os.path.exists(search_root):
            continue
        for root, _, files in os.walk(search_root):
            for f in files:
                if f.lower() == "arm_test_benchmark.parquet":
                    return os.path.join(root, f)
    return None

TEST_PATH = find_test_benchmark_parquet()
assert TEST_PATH is not None, "arm_test_benchmark.parquet not found! Attach stellar-arm-shard-data-pipeline-s1."
print(f"[TEST DATA]  {TEST_PATH}")

CHECKPOINTS = {
    "SCALAR_RM": {
        "keywords": ["scalar_rm", "scalar-rm"],
        "default_dir": "/kaggle/input/notebooks/pradeep01223/ablations-stellar-scalar-rm/checkpoints/SCALAR_RM/best",
        "associative": False, "use_dag_mixer": False, "use_monotonic": False,
        "use_lrsv_veto": False, "scalar_only_mode": True,
    },
    "INDEPENDENT_6D": {
        "keywords": ["independent_6d", "independent-6d"],
        "default_dir": "/kaggle/input/notebooks/mrsstrange012/ablations-stellar-arm-independent-6d/checkpoints/INDEPENDENT_6D/best",
        "associative": False, "use_dag_mixer": False, "use_monotonic": False,
        "use_lrsv_veto": False, "scalar_only_mode": False,
    },
    "STELLAR_NO_VETO": {
        "keywords": ["stellar_no_veto", "no_veto", "no-veto"],
        "default_dir": "/kaggle/input/notebooks/mrsstrange012/ablations-stellar-arm-no-veto/checkpoints/STELLAR_NO_VETO/best",
        "associative": True, "use_dag_mixer": True, "use_monotonic": True,
        "use_lrsv_veto": False, "scalar_only_mode": False,
    },
    "STELLAR_FULL": {
        "keywords": ["stellar_full", "stellar-shard1", "shard1"],
        "default_dir": "/kaggle/input/notebooks/rjjack0112/stellar-shard1-training-pipeline/checkpoints/STELLAR_FULL/best",
        "associative": True, "use_dag_mixer": True, "use_monotonic": True,
        "use_lrsv_veto": True, "scalar_only_mode": False,
    },
}

def resolve_checkpoint_dir(mode_name: str, cfg: Dict) -> str:
    d = cfg["default_dir"]
    if os.path.exists(os.path.join(d, "lora_weights.pt")) and os.path.exists(os.path.join(d, "arm_heads.pt")):
        return d
    candidates = []
    for search_root in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.exists(search_root):
            continue
        for root, _, files in os.walk(search_root):
            if "lora_weights.pt" in files and "arm_heads.pt" in files:
                if any(kw in root.lower() for kw in cfg["keywords"]):
                    candidates.append(root)
    if candidates:
        candidates.sort(key=lambda p: (0 if p.rstrip("/").endswith("best") else 1, len(p)))
        return candidates[0]
    raise FileNotFoundError(f"Missing checkpoint for {mode_name}!")

for name, cfg in CHECKPOINTS.items():
    cfg["dir"] = resolve_checkpoint_dir(name, cfg)
    print(f"[CKPT] {name:<16} -> {cfg['dir']}")

[BASE MODEL] /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
[TEST DATA]  /kaggle/input/notebooks/pradeep01223/stellar-arm-shard-data-pipeline-s1/arm_test_benchmark.parquet
[CKPT] SCALAR_RM        -> /kaggle/input/notebooks/pradeep01223/ablations-stellar-scalar-rm/checkpoints/SCALAR_RM/best
[CKPT] INDEPENDENT_6D   -> /kaggle/input/notebooks/mrsstrange012/ablations-stellar-arm-independent-6d/checkpoints/INDEPENDENT_6D/best
[CKPT] STELLAR_NO_VETO  -> /kaggle/input/notebooks/mrsstrange012/ablations-stellar-arm-no-veto/checkpoints/STELLAR_NO_VETO/best
[CKPT] STELLAR_FULL     -> /kaggle/input/notebooks/rjjack0112/stellar-shard1-training-pipeline/checkpoints/STELLAR_FULL/best


In [4]:
# ------------------------------------------------------------------------------
# 2. LOAD 4-BIT QWEN3-8B PEFT BACKBONE ONCE
# ------------------------------------------------------------------------------
num_gpus = torch.cuda.device_count()
assert num_gpus > 0, "No GPU detected! Enable 2x T4 GPU in Kaggle settings."
HEAD_DEVICE = torch.device(f"cuda:{num_gpus - 1}") if num_gpus > 1 else torch.device("cuda:0")
MAX_LENGTH = 1024
BATCH_SIZE = 4  # Safe for 1024-token tail batches on 2x T4; raise to 8 if VRAM allows

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|endoftext|>"
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
max_mem = {0: "8.0GiB", 1: "12.0GiB"} if num_gpus >= 2 else {0: "14.5GiB"}

base_model = AutoModel.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
    max_memory=max_mem,
    attn_implementation="eager",
    trust_remote_code=True,
)
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
peft_backbone = get_peft_model(base_model, lora_config)
peft_backbone.eval()
print(f"[BACKBONE READY] Loaded on {num_gpus} GPU(s) | Head device: {HEAD_DEVICE}", flush=True)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BACKBONE READY] Loaded on 2 GPU(s) | Head device: cuda:1


In [5]:
# ------------------------------------------------------------------------------
# 3. ARCHITECTURE DEFINITION (EXACT MATCH TO SAVED CHECKPOINTS)
# ------------------------------------------------------------------------------
class CausalDAGLayer(nn.Module):
    def __init__(self, d_model: int = 512, nhead: int = 8, dim_feedforward: int = 1024, dropout: float = 0.05):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.gate_proj = nn.Linear(d_model * 2, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, associative: bool = True) -> torch.Tensor:
        dev = z.device
        attn_mask = (
            torch.triu(torch.ones((6, 6), dtype=torch.bool, device=dev), diagonal=1)
            if associative else ~torch.eye(6, dtype=torch.bool, device=dev)
        )
        c, _ = self.self_attn(z, z, z, attn_mask=attn_mask, need_weights=False)
        g = torch.sigmoid(self.gate_proj(torch.cat([z, c], dim=-1)))
        z_mixed = (1.0 - g) * z + g * c
        z_mixed = torch.cat([z[:, :1, :], z_mixed[:, 1:, :]], dim=1)
        z_out = self.norm1(z_mixed)
        z_out = self.norm2(z_out + self.ffn(z_out))
        return torch.cat([z[:, :1, :], z_out[:, 1:, :]], dim=1)


class Qwen3AssociativeRM(nn.Module):
    GATEKEEPER_INDICES = [0, 3, 4]

    def __init__(self, peft_backbone: nn.Module, associative=True, use_monotonic=True,
                 use_lrsv_veto=True, scalar_only_mode=False, dag_num_layers=2,
                 dag_hidden_size=512, dag_num_heads=8, beta=1.0,
                 veto_kappa=1.5, monotonic_tau=0.5, monotonic_delta_tol=0.20):
        super().__init__()
        self.backbone = peft_backbone
        self.associative = associative
        self.use_monotonic = use_monotonic
        self.use_lrsv_veto = use_lrsv_veto
        self.scalar_only_mode = scalar_only_mode
        self.beta = beta
        self.veto_kappa = veto_kappa
        self.monotonic_tau = monotonic_tau
        self.monotonic_delta_tol = monotonic_delta_tol

        self.input_device = self.backbone.get_input_embeddings().weight.device
        self.head_device = HEAD_DEVICE
        self.hidden_size = self.backbone.config.hidden_size

        self.query_embeds = nn.Parameter(torch.zeros(1, 6, self.hidden_size, dtype=torch.float32, device=self.head_device))
        self.dag_attn_bias = nn.Parameter(torch.zeros((6, 6), dtype=torch.float32, device=self.head_device))
        prior = torch.tril(torch.ones((6, 6), dtype=torch.float32, device=self.head_device), diagonal=-1)
        prior[4, 3], prior[4, 2], prior[3, 2] = 2.5, 2.0, 1.5
        self.register_buffer("dag_prior", prior, persistent=False)

        self.dag_down_proj = nn.Linear(self.hidden_size, dag_hidden_size, dtype=torch.float32, device=self.head_device)
        self.dag_norm = nn.LayerNorm(dag_hidden_size, dtype=torch.float32, device=self.head_device)
        self.dag_layers = nn.ModuleList([
            CausalDAGLayer(dag_hidden_size, dag_num_heads, dag_hidden_size * 2, dropout=0.05).to(
                device=self.head_device, dtype=torch.float32
            ) for _ in range(dag_num_layers)
        ])
        self.dim_heads = nn.ModuleList([
            nn.Linear(dag_hidden_size, 1, dtype=torch.float32, device=self.head_device) for _ in range(6)
        ])
        self.agg_logits = nn.Parameter(torch.zeros(6, dtype=torch.float32, device=self.head_device))
        self.veto_thresholds = nn.Parameter(torch.zeros(len(self.GATEKEEPER_INDICES), dtype=torch.float32, device=self.head_device))
        self.veto_raw_lambda = nn.Parameter(torch.tensor(1.85, dtype=torch.float32, device=self.head_device))

    def build_4d_attention_mask(self, token_mask: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
        B, L = token_mask.shape
        T = L + 6
        dev = token_mask.device
        causal_grid = torch.tril(torch.ones((T, T), dtype=torch.bool, device=dev))
        if not self.associative:
            causal_grid[L:, L:] = torch.eye(6, dtype=torch.bool, device=dev)
        query_padding = torch.ones((B, 6), dtype=token_mask.dtype, device=dev)
        full_mask_2d = torch.cat([token_mask, query_padding], dim=1)
        valid_attn = causal_grid.unsqueeze(0).unsqueeze(0) & full_mask_2d.bool().unsqueeze(1).unsqueeze(2)
        mask_4d = torch.where(valid_attn, torch.tensor(0.0, dtype=dtype, device=dev), torch.tensor(-1e4, dtype=dtype, device=dev))

        if self.associative and (not self.scalar_only_mode):
            L_b = token_mask.float().sum(dim=-1, keepdim=True).clamp(min=1.0)
            log_L = 0.5 * torch.log(L_b).view(B, 1, 1, 1).to(device=dev, dtype=torch.float32)
            strict_tril = torch.tril(torch.ones((6, 6), dtype=torch.bool, device=dev), diagonal=-1)
            gamma_sig = torch.sigmoid(self.dag_attn_bias.to(device=dev, dtype=torch.float32))
            prior_dev = self.dag_prior.to(device=dev, dtype=torch.float32)
            q_bias = log_L * gamma_sig.unsqueeze(0).unsqueeze(0) + prior_dev.unsqueeze(0).unsqueeze(0)
            q_bias = torch.where(strict_tril.unsqueeze(0).unsqueeze(0), q_bias, torch.zeros_like(q_bias))
            mask_4d[:, :, L:, L:] = mask_4d[:, :, L:, L:] + q_bias.to(dtype=dtype)
        return mask_4d

    def forward_dag_head_only(self, last_6_hidden_fp32: torch.Tensor) -> Dict[str, torch.Tensor]:
        z = self.dag_norm(self.dag_down_proj(last_6_hidden_fp32))
        for layer in self.dag_layers:
            z = layer(z, associative=self.associative)
        raw_z = torch.cat([self.dim_heads[i](z[:, i, :]) for i in range(6)], dim=-1)
        z_clamped = torch.clamp(raw_z, min=-15.0, max=15.0)

        if self.scalar_only_mode:
            agg_weights = F.softmax(self.agg_logits, dim=0)
            return {
                "scalar_logit": (z_clamped * agg_weights).sum(dim=-1),
                "dim_scores": torch.sigmoid(z_clamped),
            }

        if self.associative and self.use_monotonic:
            z3, z4, tau_m = z_clamped[:, 2:3], z_clamped[:, 3:4], self.monotonic_tau
            z_prereq = -tau_m * torch.logsumexp(torch.cat([-z3 / tau_m, -z4 / tau_m], dim=-1), dim=-1, keepdim=True) + tau_m * math.log(2.0)
            z5_mono = torch.clamp(z_prereq + self.monotonic_delta_tol - F.softplus(-raw_z[:, 4:5]), min=-15.0, max=15.0)
            dim_logits = torch.cat([z_clamped[:, :4], z5_mono, z_clamped[:, 5:]], dim=-1)
        else:
            dim_logits = z_clamped

        agg_weights = F.softmax(self.agg_logits, dim=0)
        linear_term = self.beta * (dim_logits * agg_weights).sum(dim=-1)
        if self.use_lrsv_veto:
            gk_logits = dim_logits[:, self.GATEKEEPER_INDICES]
            veto_lambda = F.softplus(self.veto_raw_lambda)
            exponents = -self.veto_kappa * (gk_logits - self.veto_thresholds.unsqueeze(0))
            zero_pad = torch.zeros((exponents.size(0), 1), dtype=exponents.dtype, device=exponents.device)
            veto_penalty = (veto_lambda / self.veto_kappa) * torch.logsumexp(torch.cat([zero_pad, exponents], dim=-1), dim=-1)
            scalar_logit = linear_term - veto_penalty
        else:
            scalar_logit = linear_term

        return {"scalar_logit": scalar_logit, "dim_scores": torch.sigmoid(dim_logits)}

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> Dict[str, torch.Tensor]:
        B = input_ids.shape[0]
        input_ids = input_ids.to(self.input_device)
        attention_mask = attention_mask.to(self.input_device)
        tok_embeds = self.backbone.get_input_embeddings()(input_ids)
        q_exp = self.query_embeds.to(device=self.input_device, dtype=tok_embeds.dtype).expand(B, -1, -1)
        inputs_embeds = torch.cat([tok_embeds, q_exp], dim=1)
        full_mask_2d = torch.cat([attention_mask, torch.ones((B, 6), dtype=attention_mask.dtype, device=self.input_device)], dim=1)
        pos_ids = (full_mask_2d.long().cumsum(-1) - 1).clamp(min=0)
        mask_4d = self.build_4d_attention_mask(attention_mask, dtype=tok_embeds.dtype)

        outputs = self.backbone(inputs_embeds=inputs_embeds, attention_mask=mask_4d, position_ids=pos_ids, return_dict=True)
        last_6_fp32 = outputs.last_hidden_state[:, -6:, :].to(device=self.head_device, dtype=torch.float32)
        with torch.cuda.device(self.head_device):
            return self.forward_dag_head_only(last_6_fp32)

    def load_head_state_dict(self, state: Dict[str, Any]):
        self.query_embeds.data.copy_(state["query_embeds"].to(self.head_device))
        self.agg_logits.data.copy_(state["agg_logits"].to(self.head_device))
        if "dag_attn_bias" in state:
            self.dag_attn_bias.data.copy_(state["dag_attn_bias"].to(self.head_device))
            self.dag_down_proj.load_state_dict(state["dag_down_proj"])
            self.dag_norm.load_state_dict(state["dag_norm"])
            if len(self.dag_layers) > 0 and "dag_layers" in state:
                self.dag_layers.load_state_dict(state["dag_layers"])
            self.dim_heads.load_state_dict(state["dim_heads"])
            self.veto_thresholds.data.copy_(state["veto_thresholds"].to(self.head_device))
            self.veto_raw_lambda.data.copy_(state["veto_raw_lambda"].to(self.head_device))
        self.associative = state.get("associative", self.associative)
        self.use_monotonic = state.get("use_monotonic", self.use_monotonic)
        self.use_lrsv_veto = state.get("use_lrsv_veto", self.use_lrsv_veto)
        self.beta = state.get("beta", self.beta)
        self.scalar_only_mode = state.get("scalar_only_mode", self.scalar_only_mode)

In [6]:
# ------------------------------------------------------------------------------
# 4. PRE-TOKENIZE ONCE & FAST LENGTH-SORTED SCORER
# ------------------------------------------------------------------------------
def truncate_prompt_response(prompt: str, response: str, tok, max_len=1024, resp_reserve=640) -> List[int]:
    p_ids = tok.encode(prompt, add_special_tokens=False)
    r_ids = tok.encode(response, add_special_tokens=False)
    if len(p_ids) + len(r_ids) <= max_len:
        return p_ids + r_ids
    r_keep = r_ids[-min(len(r_ids), resp_reserve):]
    p_keep = p_ids[-(max_len - len(r_keep)):] if (max_len - len(r_keep)) > 0 else []
    return p_keep + r_keep

df_test = pd.read_parquet(TEST_PATH).reset_index(drop=True)
prompts = df_test["prompt"].tolist()
chosen = df_test["chosen"].tolist()
rejected = df_test["rejected"].tolist()
regimes = np.array(df_test["regime"].tolist())

padded_rejected = [
    "Certainly! Here is a clear, well-structured breakdown of the solution:\n\n"
    f"**Step-by-Step Analysis:**\n{r}\n\n"
    "**Summary:** Following this structured reasoning ensures a thorough and accurate answer."
    for r in rejected
]

print("[TOKENIZING] Pre-tokenizing chosen, rejected, and style-padded splits once...", flush=True)
seqs_chosen   = [truncate_prompt_response(p, r, tokenizer, MAX_LENGTH, 640) for p, r in zip(prompts, chosen)]
seqs_rejected = [truncate_prompt_response(p, r, tokenizer, MAX_LENGTH, 640) for p, r in zip(prompts, rejected)]
seqs_padded   = [truncate_prompt_response(p, r, tokenizer, MAX_LENGTH, 640) for p, r in zip(prompts, padded_rejected)]
print(f"[TOKENIZING] Done ({len(seqs_chosen)} pairs).", flush=True)

@torch.inference_mode()
def score_pretokenized(rm_model, seqs: List[List[int]], label: str = "", batch_size: int = BATCH_SIZE):
    rm_model.eval()
    n = len(seqs)
    order = np.argsort([len(s) for s in seqs])
    rewards = np.zeros(n, dtype=np.float64)
    dims = np.zeros((n, 6), dtype=np.float64)
    pad_id = tokenizer.pad_token_id
    t0 = time.time()

    for start in range(0, n, batch_size):
        idx_batch = order[start:start + batch_size]
        batch_seqs = [seqs[i] for i in idx_batch]
        m_len = max(len(s) for s in batch_seqs)

        ids = torch.full((len(batch_seqs), m_len), pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch_seqs), m_len), dtype=torch.long)
        for row, s in enumerate(batch_seqs):
            ids[row, m_len - len(s):] = torch.tensor(s, dtype=torch.long)
            mask[row, m_len - len(s):] = 1

        out = rm_model(ids, mask)
        rewards[idx_batch] = out["scalar_logit"].detach().cpu().numpy()
        dims[idx_batch] = out["dim_scores"].detach().cpu().numpy()

        done = min(start + batch_size, n)
        if (start // batch_size) % 15 == 0 or done == n:
            elapsed = time.time() - t0
            rate = done / max(elapsed, 1e-6)
            eta = (n - done) / max(rate, 1e-6)
            print(f"  [{label}] {done}/{n} ({rate:.2f} seq/s | ETA: {eta/60:.1f}m)", flush=True)

    return rewards, dims

[TOKENIZING] Pre-tokenizing chosen, rejected, and style-padded splits once...
[TOKENIZING] Done (750 pairs).


In [7]:
# ------------------------------------------------------------------------------
# 5. RUN 4-WAY AUDIT & SAVE RESULTS
# ------------------------------------------------------------------------------
obj_mask = np.isin(regimes, ["R2_math_reasoning", "R3_code_mbpp_ast_bugs", "R4_ifeval_constraints"])
r6_mask = (regimes == "R6_off_diagonal_ugly_vs_fluent")
r7_mask = (regimes == "R7_safety_refusal")

all_results = {}

for mode_name, cfg in CHECKPOINTS.items():
    print(f"\n{'='*80}\n[AUDITING] {mode_name}\n{'='*80}", flush=True)
    set_peft_model_state_dict(
        peft_backbone,
        torch.load(os.path.join(cfg["dir"], "lora_weights.pt"), map_location="cpu", weights_only=False),
    )
    rm = Qwen3AssociativeRM(
        peft_backbone=peft_backbone,
        associative=cfg["associative"],
        use_monotonic=cfg["use_monotonic"],
        use_lrsv_veto=cfg["use_lrsv_veto"],
        scalar_only_mode=cfg["scalar_only_mode"],
        dag_num_layers=2 if cfg["use_dag_mixer"] else 0,
    )
    rm.load_head_state_dict(
        torch.load(os.path.join(cfg["dir"], "arm_heads.pt"), map_location=HEAD_DEVICE, weights_only=False)
    )
    rm.eval()

    r_c, d_c = score_pretokenized(rm, seqs_chosen,   label=f"{mode_name}:chosen")
    r_r, d_r = score_pretokenized(rm, seqs_rejected, label=f"{mode_name}:rejected")
    r_p, _   = score_pretokenized(rm, seqs_padded,   label=f"{mode_name}:padded")

    base_ok = r_c > r_r
    pad_ok = r_c > r_p
    flipped = base_ok & (~pad_ok)

    obj_base_ok = base_ok[obj_mask]
    obj_flipped = flipped[obj_mask]

    veto_lam = float(F.softplus(rm.veto_raw_lambda).item()) if rm.use_lrsv_veto else 0.0
    all_dims = np.concatenate([d_c, d_r], axis=0)
    level_c_viol = float((all_dims[:, 4] > (all_dims[:, 3] + 0.10)).mean()) * 100.0

    res = {
        "overall_acc_pct": round(float(base_ok.mean()) * 100, 2),
        "r6_style_trap_acc_pct": round(float(base_ok[r6_mask].mean()) * 100, 2),
        "r6_mean_margin": round(float((r_c[r6_mask] - r_r[r6_mask]).mean()), 4),
        "r7_safety_acc_pct": round(float(base_ok[r7_mask].mean()) * 100, 2),
        "padded_overall_acc_pct": round(float(pad_ok.mean()) * 100, 2),
        "overall_style_flip_rate_pct": round(float(flipped.sum() / max(1, base_ok.sum())) * 100, 2),
        "objective_style_flip_rate_pct": round(float(obj_flipped.sum() / max(1, obj_base_ok.sum())) * 100, 2),
        "mean_fluff_reward_boost": round(float((r_p - r_r).mean()), 4),
        "level_c_viol_pct": None if cfg["scalar_only_mode"] else round(level_c_viol, 2),
        "veto_lambda": round(veto_lam, 3),
    }
    all_results[mode_name] = res
    print(json.dumps(res, indent=2), flush=True)

    del rm
    torch.cuda.empty_cache()
    gc.collect()


[AUDITING] SCALAR_RM
  [SCALAR_RM:chosen] 4/750 (4.05 seq/s | ETA: 3.1m)
  [SCALAR_RM:chosen] 64/750 (9.79 seq/s | ETA: 1.2m)
  [SCALAR_RM:chosen] 124/750 (8.84 seq/s | ETA: 1.2m)
  [SCALAR_RM:chosen] 184/750 (8.13 seq/s | ETA: 1.2m)
  [SCALAR_RM:chosen] 244/750 (7.40 seq/s | ETA: 1.1m)
  [SCALAR_RM:chosen] 304/750 (6.74 seq/s | ETA: 1.1m)
  [SCALAR_RM:chosen] 364/750 (6.16 seq/s | ETA: 1.0m)
  [SCALAR_RM:chosen] 424/750 (5.61 seq/s | ETA: 1.0m)
  [SCALAR_RM:chosen] 484/750 (5.14 seq/s | ETA: 0.9m)
  [SCALAR_RM:chosen] 544/750 (4.63 seq/s | ETA: 0.7m)
  [SCALAR_RM:chosen] 604/750 (4.12 seq/s | ETA: 0.6m)
  [SCALAR_RM:chosen] 664/750 (3.65 seq/s | ETA: 0.4m)
  [SCALAR_RM:chosen] 724/750 (3.15 seq/s | ETA: 0.1m)
  [SCALAR_RM:chosen] 750/750 (2.86 seq/s | ETA: 0.0m)
  [SCALAR_RM:rejected] 4/750 (12.95 seq/s | ETA: 1.0m)
  [SCALAR_RM:rejected] 64/750 (9.87 seq/s | ETA: 1.2m)
  [SCALAR_RM:rejected] 124/750 (8.53 seq/s | ETA: 1.2m)
  [SCALAR_RM:rejected] 184/750 (7.71 seq/s | ETA: 1.2m)
  [

In [8]:
# ------------------------------------------------------------------------------
# 6. PRINT MARKDOWN TABLE & SAVE JSON
# ------------------------------------------------------------------------------
print("\n\n### 4-WAY FAILURE-MODE AUDIT SUMMARY")
print("| Model | Test Acc | R6 Style Trap | R6 Margin | R7 Safety | Padded Acc | Style Flip (All) | Style Flip (Math/Code/IF) | Fluff Boost | Veto λ |")
print("| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |")
for name, r in all_results.items():
    print(
        f"| **{name}** | {r['overall_acc_pct']}% | {r['r6_style_trap_acc_pct']}% | "
        f"{r['r6_mean_margin']:+.3f} | {r['r7_safety_acc_pct']}% | {r['padded_overall_acc_pct']}% | "
        f"{r['overall_style_flip_rate_pct']}% | {r['objective_style_flip_rate_pct']}% | "
        f"{r['mean_fluff_reward_boost']:+.3f} | {r['veto_lambda']:.3f} |"
    )

out_path = "/kaggle/working/failure_mode_audit_results.json"
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\n[SAVED] {out_path}")



### 4-WAY FAILURE-MODE AUDIT SUMMARY
| Model | Test Acc | R6 Style Trap | R6 Margin | R7 Safety | Padded Acc | Style Flip (All) | Style Flip (Math/Code/IF) | Fluff Boost | Veto λ |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **SCALAR_RM** | 92.67% | 95.0% | +7.263 | 96.0% | 95.6% | 0.86% | 0.0% | -0.909 | 0.000 |
| **INDEPENDENT_6D** | 93.73% | 95.0% | +2.085 | 97.0% | 95.6% | 0.85% | 0.0% | -0.110 | 0.000 |
| **STELLAR_NO_VETO** | 93.73% | 95.0% | +1.973 | 96.0% | 95.07% | 1.0% | 0.0% | -0.086 | 0.000 |
| **STELLAR_FULL** | 92.93% | 94.0% | +5.034 | 98.0% | 97.6% | 0.29% | 0.0% | +0.024 | 2.020 |

[SAVED] /kaggle/working/failure_mode_audit_results.json
